# 3D U-Net Bone & Edema Training (Kaggle Cloud GPU & Colab)

> ⚠️ **CLINICAL DATA GOVERNANCE & PRIVACY NOTICE**  
> Train strictly on **fully de-identified, anonymized research data** conforming to HIPAA/IRB ethics protocols.  
> Never upload raw patient identifiers (PHI/PII), MRNs, accession numbers, or unscrubbed DICOM tags.

This notebook trains a 3D U-Net on MRI patches (`96×96×12`) for joint bone and bone marrow edema (BME) segmentation using patient-level cross-validation.

---

### 🚀 Kaggle Execution Steps (Free NVIDIA T4 GPU — Zero Local Setup)
1. **Upload Notebook**:
   - On [Kaggle](https://www.kaggle.com), create a new notebook or click **File > Upload Notebook** and select this file (`train_3d_colab.ipynb`).
2. **Attach Private Dataset**:
   - In the right-hand panel under **Input**, click **+ Add Input**.
   - Click **Upload** and upload your exported dataset archive (e.g. `bme_dataset_bundle.zip` or `bme_colab_data.zip`).
   - Give it any title (e.g. `bme-3d-dataset`), ensure it is marked **Private Dataset**, and click **Create**.
3. **Turn GPU On**:
   - In the right sidebar: expand **Settings** > **Accelerator** > select **GPU T4 x2** or **GPU T4** (30 free GPU hours every week).
4. **Turn Internet On**:
   - In the right sidebar: expand **Settings** > **Internet** > toggle **Internet on** (required for `pip install nibabel`).
5. **Start Cloud Background Execution**:
   - Click **Save Version** (top right) > select **Save & Run All (Commit)**.
   - You can now safely close your browser and turn off your laptop — Kaggle runs and completes training independently in Google's cloud.

---

### 📦 Expected Outputs
All checkpoints and metrics are written to `/kaggle/working/results3d/`:
- `fold_0.pt` … `fold_4.pt`: Trained 3D model weights for each cross-validation fold.
- `metrics.json`: Held-out validation metrics (Bone Dice, BME Dice, lesion sensitivity, false positive counts).
- `results3d.zip`: Self-contained archive packaged in `/kaggle/working/` containing all weights and metrics, ready for download in the notebook's **Output** tab.

---

### 📥 Importing Weights Back into the BME App
1. When training finishes, navigate to the notebook's **Output** tab and download **`results3d.zip`**.
2. Open your local BME app and navigate to **Training > 3D U-Net**.
3. Under the **Import Weights** card, drag & drop `results3d.zip` (or click Browse) and click **Import Weights**.
4. The app verifies the model checkpoints, backs up any previous weights, and immediately updates the results dashboard and inference models!

---

### 🔄 Resuming an Interrupted Run
If a Kaggle session reaches its time limit, you can resume seamlessly without re-training finished folds:
1. In your new Kaggle notebook version, click **+ Add Input** > **Your Work / Notebook Output** > select the previous version.
2. Keep `RESUME = True` in the Configuration cell.
3. The notebook will automatically discover the previous checkpoints in `/kaggle/input/`, copy them into `/kaggle/working/results3d/`, skip completed folds, and resume training!


## 1. GPU Hardware & PyTorch Diagnostics
Inspect GPU allocation (NVIDIA T4 / P100), VRAM, and PyTorch CUDA build.

In [ ]:
import os
import sys
import torch

print("=" * 60)
print("GPU HARDWARE & PYTORCH DIAGNOSTICS")
print("=" * 60)
print(f"Python version  : {sys.version.split()[0]}")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Active GPU      : {props.name}")
    print(f"Total VRAM      : {props.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability : {props.major}.{props.minor}")
    print(f"PyTorch CUDA ver: {torch.version.cuda}")
    print("=" * 60)
    !nvidia-smi
else:
    print("=" * 60)
    print("⚠️ WARNING: No CUDA GPU detected!")
    print("If running on Kaggle: Settings (right sidebar) -> Accelerator -> select GPU T4.")
    print("If running on Colab: Runtime -> Change runtime type -> T4 GPU.")


## 2. Configuration & Hyperparameters
Select your training configuration mode and storage paths:
- **`CONFIG_MODE = "quick_test"` (DEFAULT)**: Fast smoke test (2 epochs, 2 folds, 4 cases) to verify the environment before a full run.
- **`CONFIG_MODE = "full"`**: Full 40-epoch, 5-fold cross-validation across all dataset cases.


In [ ]:
# =====================================================================
# 1. RUN MODE: 'quick_test' (default) | 'bench' | 'fold0' | 'full'
# =====================================================================
CONFIG_MODE = "quick_test"  # Options: 'bench', 'quick_test', 'fold0', 'full', 'fold0_ablation', 'eval_suite'
MODEL_ARCH = "mednext"  # choices: "mednext", "cnet_attention_unet", "unet3d"
RUN_EVAL = False  # Safety guard: must be explicitly set to True when CONFIG_MODE == 'eval_suite'
RUN_ABLATION = False  # Safety guard: must be explicitly set to True when CONFIG_MODE == 'fold0_ablation'

# =====================================================================
# 2. PLATFORM TARGET: 'kaggle' (default) | 'colab' | 'standalone'
# =====================================================================
PLATFORM = "kaggle"

# =====================================================================
# 3. KAGGLE PATH CONFIGURATION (Read-only input -> Writable /kaggle/working)
# =====================================================================
# Optional: Specific dataset slug under /kaggle/input/ (leave "" for auto-discovery)
KAGGLE_DATASET_NAME = ""

# Optional: Slug of a previous notebook run added as input to resume from (leave "" for auto-discovery)
KAGGLE_RESUME_INPUT = ""

# Writable directories on Kaggle fast local storage
KAGGLE_WORKING_DIR = "/kaggle/working"
KAGGLE_OUTPUT_DIR = "/kaggle/working/results3d"

# =====================================================================
# 4. COLAB & STANDALONE PATH CONFIGURATION
# =====================================================================
COLAB_DRIVE_FOLDER = "/content/drive/MyDrive/bme_project"
COLAB_OUTPUT_DIR = "/content/drive/MyDrive/bme_project/data/results3d"
STANDALONE_DIR = "."
STANDALONE_OUTPUT_DIR = "./data/results3d"

# =====================================================================
# 5. HYPERPARAMETERS FOR PRODUCTION TRAINING (SCIENTIFIC BASELINE)
# =====================================================================
FULL_EPOCHS = 40
FULL_FOLDS = 5
RUN_FOLDS = None  # None for all folds [0, 1, 2, 3, 4], or list e.g. [0] or [0, 1, 2]
SPECIFIC_FOLD = None  # Backward-compatible alias for single fold

# =====================================================================
# 6. TIME GUARD, REUSE, & SCHEDULER
# =====================================================================
MAX_RUNTIME_MINUTES = None  # Default None (off). Set e.g. 540 for 9h Kaggle limit, or 60 for 1h test
SKIP_COMPLETED = True       # Skip folds already completed in attached output if config/seed match
FORCE_SEQUENTIAL = False    # Set to True to disable parallel T4 x2 scheduler and force sequential

# =====================================================================
# 7. OPTIMIZATION FLAGS (Scientifically Equivalent, Identical Experiment)
# =====================================================================
USE_AMP = True                  # Automatic Mixed Precision (FP16) on GPU
USE_CUDNN_BENCHMARK = True      # Fast cuDNN auto-tuner for static patch shapes
USE_CHANNELS_LAST = True        # channels_last_3d memory format (faster 3D convs)
VAL_BATCH_SIZE = 8              # Batch size for sliding-window validation
PROFILE = True                  # Per-epoch profiling breakdown (data, forward/backward, val, save)
RESUME = True                   # Automatically skips completed folds / resumes latest epoch
CHECKPOINT_EVERY_EPOCH = True   # Saves checkpoint after each epoch to prevent lost work

# =====================================================================
# 8. DATA QUALITY & ACCURACY ENHANCEMENTS (BME LESION FOCUS)
# =====================================================================
ENABLE_DATA_AUGMENTATION = True     # 3D spatial flip, intensity scaling, and noise
RESAMPLE_EACH_EPOCH = True          # Dynamic patch resampling each epoch in memory
SAMPLING_MIX = "lesion=0.5,bone=0.35,random=0.15"  # Balanced patch sampling (50% lesion, 35% bone, 15% random)

print("=" * 60)
print(f"Active Mode       : {CONFIG_MODE.upper()}")
print(f"Run Ablation Guard: {RUN_ABLATION}")
print(f"Platform Target   : {PLATFORM.upper()}")
print(f"Run Folds         : {RUN_FOLDS if RUN_FOLDS is not None else 'All 5 Folds'}")
print(f"Skip Completed    : {SKIP_COMPLETED}")
print(f"Time Guard        : {f'{MAX_RUNTIME_MINUTES} min' if MAX_RUNTIME_MINUTES else 'Disabled'}")
print(f"Force Sequential  : {FORCE_SEQUENTIAL}")
print(f"Auto-Resume       : {RESUME}")
print(f"Mixed Precision   : {USE_AMP}")
print(f"cuDNN Benchmark   : {USE_CUDNN_BENCHMARK}")
print(f"Channels Last 3D  : {USE_CHANNELS_LAST}")
print(f"Val Batch Size    : {VAL_BATCH_SIZE}")
print(f"Profile Timing    : {PROFILE}")
print(f"Checkpoint/Epoch  : {CHECKPOINT_EVERY_EPOCH}")
print(f"Data Augmentation : {ENABLE_DATA_AUGMENTATION}")
print(f"Resample/Epoch    : {RESAMPLE_EACH_EPOCH}")
print(f"Sampling Mix      : {SAMPLING_MIX}")
print("=" * 60)


## 3. Install Dependencies
Installs medical imaging and numerical packages (`nibabel` for NIfTI volumes, `scipy`, `pynrrd`).

In [ ]:
import sys
import subprocess
import time

t_install_start = time.time()
packages_to_check = [
    ("nibabel", "nibabel"),
    ("scipy", "scipy"),
    ("SimpleITK", "SimpleITK"),
    ("pynrrd", "nrrd"),
]

missing_pkgs = []
for pkg_name, mod_name in packages_to_check:
    try:
        __import__(mod_name)
    except ImportError:
        missing_pkgs.append(pkg_name)

if missing_pkgs:
    print(f"Installing missing dependencies: {', '.join(missing_pkgs)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs, check=True)
    print("✓ Installation complete.")
else:
    print("✓ All required dependencies already available (no pip install needed).")

t_install_time = time.time() - t_install_start
print(f"Dependency check time: {t_install_time:.2f}s")


## 4. Dataset Discovery & Previous Run Checkpoint Pickup
Locates the BME dataset under `/kaggle/input` (read-only), unpacks any archive if needed to fast SSD, validates data integrity, and copies previous checkpoints to `/kaggle/working/results3d` for seamless resuming.


In [ ]:
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

t_discovery_start = time.time()

# Resolve Output Directory (Strictly in /kaggle/working for Kaggle)
if PLATFORM == "kaggle":
    OUT_DIR = Path(KAGGLE_OUTPUT_DIR)
elif PLATFORM == "colab":
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount('/content/drive')
    OUT_DIR = Path(COLAB_OUTPUT_DIR)
else:
    OUT_DIR = Path(STANDALONE_OUTPUT_DIR).resolve()

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# A. CHECKPOINT RESUME FROM PREVIOUS NOTEBOOK RUN (If added as input)
# ---------------------------------------------------------------------
if RESUME and PLATFORM == "kaggle":
    input_base = Path("/kaggle/input")
    resume_candidates = []
    
    if KAGGLE_RESUME_INPUT:
        specific = input_base / KAGGLE_RESUME_INPUT
        if specific.is_dir():
            resume_candidates.append(specific)
            
    # Auto-discover previous notebook output in /kaggle/input
    if input_base.is_dir():
        for item in input_base.iterdir():
            if item.is_dir() and item not in resume_candidates:
                if list(item.glob("*.pt")) or list(item.glob("results3d/*.pt")):
                    resume_candidates.append(item)

    copied_checkpoints = 0
    for cand in resume_candidates:
        src_pt_files = list(cand.glob("*.pt")) + list(cand.glob("results3d/*.pt"))
        src_json_files = list(cand.glob("*.json")) + list(cand.glob("results3d/*.json"))
        for f in src_pt_files + src_json_files:
            dst = OUT_DIR / f.name
            if not dst.exists() or dst.stat().st_size == 0:
                shutil.copy2(f, dst)
                copied_checkpoints += 1
                
    if copied_checkpoints > 0:
        print(f"✓ Resumed: Copied {copied_checkpoints} previous checkpoints/metrics into {OUT_DIR}")
        existing_folds = sorted([p.name for p in OUT_DIR.glob("fold_*.pt")])
        print(f"  Existing completed folds: {existing_folds}")
    else:
        print("  No previous notebook checkpoints found. Starting clean run.")

# ---------------------------------------------------------------------
# B. DATASET DISCOVERY UNDER /kaggle/input (Read-only)
# ---------------------------------------------------------------------
BASE_DIR = None

if PLATFORM == "kaggle":
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        raise FileNotFoundError(
            "[ERROR] /kaggle/input does not exist! Ensure this notebook is running in Kaggle."
        )

    # 1. Check specified dataset folder if provided
    search_dirs = []
    if KAGGLE_DATASET_NAME:
        named_path = kaggle_input / KAGGLE_DATASET_NAME
        if named_path.is_dir():
            search_dirs.append(named_path)

    # 2. Add all subdirectories in /kaggle/input
    for d in sorted(kaggle_input.iterdir()):
        if d.is_dir() and d not in search_dirs:
            search_dirs.append(d)

    print(f"Scanning /kaggle/input for BME dataset ({len(search_dirs)} input directories attached)...")

    # Look for dataset layout: prioritize uncompressed folder FIRST across all search dirs
    found_dataset_path = None
    for d in search_dirs:
        if (d / "data" / "annotations").is_dir() or (d / "annotations").is_dir():
            found_dataset_path = d
            break
        for sub in d.glob("*"):
            if sub.is_dir() and ((sub / "data" / "annotations").is_dir() or (sub / "annotations").is_dir()):
                found_dataset_path = sub
                break
        if found_dataset_path:
            break

    found_zip_path = None
    if not found_dataset_path:
        for d in search_dirs:
            zips = list(d.glob("*.zip"))
            if zips:
                found_zip_path = zips[0]
                break

    if found_dataset_path:
        BASE_DIR = found_dataset_path
        print(f"✓ Found uncompressed dataset at: {BASE_DIR} (direct access, zero extraction time)")
    elif found_zip_path:
        extract_dest = Path("/tmp/bme_dataset") if PLATFORM == "kaggle" else (OUT_DIR / "bme_dataset")
        extract_dest.mkdir(parents=True, exist_ok=True)
        print(f"✓ Found dataset archive: {found_zip_path.name}")
        print(f"  Unpacking to fast local storage ({extract_dest})...")
        with zipfile.ZipFile(found_zip_path, "r") as zf:
            zf.extractall(str(extract_dest))
        # Resolve extracted path
        BASE_DIR = extract_dest
        for candidate in [extract_dest, extract_dest / "data", extract_dest / "bme_dataset_bundle"]:
            if (candidate / "data" / "annotations").is_dir() or (candidate / "annotations").is_dir():
                BASE_DIR = candidate
                break
    else:
        checked_list = [f"- {d.name}" for d in search_dirs]
        raise FileNotFoundError(
            "\n" + "=" * 70 + "\n"
            "[ERROR] No valid BME dataset found under /kaggle/input!\n"
            "=" * 70 + "\n"
            "Please attach your dataset to this Kaggle notebook:\n"
            "  1. In the right panel, click '+ Add Input'.\n"
            "  2. Click 'Upload' -> upload your exported dataset folder or zip (from the BME app).\n"
            "  3. Set it as a Private Dataset (e.g. named 'bme-3d-dataset').\n"
            "  4. Click 'Create' and ensure it appears under /kaggle/input/.\n\n"
            "Directories currently attached:\n" +
            ("\n".join(checked_list) if checked_list else "  (None)") + "\n"
            "=" * 70
        )

elif PLATFORM == "colab":
    # Colab resolution logic
    zip_candidates = [
        Path("/content/drive/MyDrive/bme_colab_data.zip"),
        Path(COLAB_DRIVE_FOLDER) / "bme_colab_data.zip",
        Path("/content/bme_colab_data.zip"),
    ]
    found_zip = next((z for z in zip_candidates if z.exists()), None)
    if found_zip:
        with zipfile.ZipFile(found_zip, "r") as zf:
            zf.extractall("/content/bme_project")
        BASE_DIR = Path("/content/bme_project")
    else:
        BASE_DIR = Path(COLAB_DRIVE_FOLDER)
else:
    BASE_DIR = Path(STANDALONE_DIR).resolve()

# ---------------------------------------------------------------------
# C. VALIDATE DATASET CONTENTS & CASE COUNTS (Anonymized)
# ---------------------------------------------------------------------
ann_dir = None
for candidate in [
    BASE_DIR / "data" / "annotations",
    BASE_DIR / "annotations",
    BASE_DIR / "bme_dataset_bundle" / "data" / "annotations",
]:
    if candidate.is_dir():
        ann_dir = candidate
        break

if ann_dir is None:
    cands = [p for p in BASE_DIR.rglob("annotations") if p.is_dir()]
    ann_dir = cands[0] if cands else None

if ann_dir is None or not ann_dir.is_dir():
    raise FileNotFoundError(f"[ERROR] Could not locate 'annotations' directory inside {BASE_DIR}.")

case_ids = []
for d in ann_dir.iterdir():
    if d.is_dir() and (list(d.glob("*_labels.nii*")) or list(d.glob("*.seg.nrrd"))):
        case_ids.append(d.name)
if not case_ids:
    all_labs = list(BASE_DIR.rglob("*_labels.nii*")) + list(BASE_DIR.rglob("*.seg.nrrd"))
    case_ids = sorted(list({p.parent.name for p in all_labs if p.is_file()}))

total_cases = len(case_ids)
if total_cases < 2:
    raise ValueError(f"[ERROR] Need at least 2 annotated volumes to train. Found {total_cases}.")

t_discovery_time = time.time() - t_discovery_start

print("\n" + "=" * 60)
print("DATASET VERIFICATION SUMMARY")
print("=" * 60)
print(f"Dataset root location : {BASE_DIR}")
print(f"Output / Checkpoints  : {OUT_DIR}")
print(f"Annotated cases found : {total_cases} cases")
print(f"Discovery wall-clock  : {t_discovery_time:.2f}s")
print("✓ Dataset layout validated successfully!")
print("=" * 60)


## 5. Self-Contained Training Script Setup
Verifies the training script or extracts the self-contained `train_3d.py` into `/kaggle/working/train_3d.py`. This guarantees the notebook runs out-of-the-box with zero external code dependencies.


In [ ]:
import base64
import hashlib
from pathlib import Path

# Embedded standalone training script
BUNDLED_CODE_B64 = "..."

# Automatically select execution script:
# 1. If train_3d.py exists alongside dataset or in repo, use that directly.
# 2. Otherwise, extract embedded BUNDLED_CODE_B64 to /kaggle/working/train_3d.py.
working_script = Path("/kaggle/working/train_3d.py") if PLATFORM == "kaggle" else (OUT_DIR / "train_3d.py")
source_script = None
for cand in [
    BASE_DIR / "data" / "train_3d.py",
    BASE_DIR / "train_3d.py",
    Path("ml/scripts/train_3d.py").resolve(),
    Path("/kaggle/working/train_3d.py"),
]:
    if cand.is_file():
        source_script = cand
        break

embedded_bytes = base64.b64decode(BUNDLED_CODE_B64)
embedded_sha256 = hashlib.sha256(embedded_bytes).hexdigest()
print("=" * 60)
print(f"Embedded train_3d.py SHA-256 : {embedded_sha256}")

if source_script and source_script.is_file():
    disk_bytes = source_script.read_bytes()
    disk_sha256 = hashlib.sha256(disk_bytes).hexdigest()
    print(f"Disk train_3d.py SHA-256     : {disk_sha256} (from {source_script})")
    if disk_sha256 != embedded_sha256:
        print("[NOTICE] Disk script differs from embedded script. Using disk version.")
    else:
        print("[OK] Embedded and disk scripts match byte-for-byte!")
    if source_script != working_script:
        import shutil
        shutil.copy2(source_script, working_script)
        print(f"[OK] Copied {source_script} -> {working_script}")
elif working_script.is_file():
    disk_bytes = working_script.read_bytes()
    disk_sha256 = hashlib.sha256(disk_bytes).hexdigest()
    print(f"Existing working script SHA-256: {disk_sha256}")
else:
    working_script.parent.mkdir(parents=True, exist_ok=True)
    working_script.write_bytes(embedded_bytes)
    print(f"[OK] Deployed self-contained train_3d.py ({len(embedded_bytes)} bytes) to {working_script}")
print("=" * 60)


## 6. Execute 3D U-Net Training
Runs training with:
- Automatic GPU detection (`--device cuda`)
- Automatic Mixed Precision (`--amp`)
- Checkpointing per epoch (`--checkpoint-every-epoch`)
- Resuming completed folds (`--resume`)
- Real-time time per epoch and total ETA


In [ ]:
import os
import json
import re
import subprocess
import sys
import time
from pathlib import Path
import torch

script_to_run = working_script if working_script.exists() else source_script
if not script_to_run or not script_to_run.exists():
    raise FileNotFoundError("[ERROR] Could not locate train_3d.py to execute.")

n_gpus = torch.cuda.device_count()
print(f"[HARDWARE] Detected {n_gpus} CUDA GPU(s). Active mode: {CONFIG_MODE.upper()}")

# =====================================================================
# A. BENCHMARK MODE ('bench'): 1 Fold x 2 Epochs across 3 Configurations
# =====================================================================
if CONFIG_MODE == "bench":
    bench_dir = Path("/kaggle/working/benchmark_results") if PLATFORM == "kaggle" else (OUT_DIR / "benchmark_results")
    bench_dir.mkdir(parents=True, exist_ok=True)

    configs = [
        ("Baseline (FP32, val_bs=1)", ["--val-batch-size", "1"]),
        ("+AMP + cuDNN + Val-Batching", ["--amp", "--cudnn-benchmark", "--val-batch-size", "8"]),
        ("+Channels-Last-3D on top", ["--amp", "--cudnn-benchmark", "--channels-last-3d", "--val-batch-size", "8"]),
    ]

    bench_results = []
    print("=" * 85)
    print(f"RUNNING GPU SPEEDUP BENCHMARK (1 Fold x 2 Epochs, Device: {'CUDA' if torch.cuda.is_available() else 'CPU'})")
    print("=" * 85)

    for name, flags in configs:
        sub_dir = bench_dir / name.replace(" ", "_").replace("+", "p_").replace("(", "").replace(")", "").replace("=", "_").replace(",", "_")
        sub_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable, "-u", str(script_to_run), str(BASE_DIR),
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
            "--out-dir", str(sub_dir),
            "--epochs", "2",
            "--folds", "5",
            "--fold", "0",
            "--profile",
        ] + flags

        t0 = time.time()
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        elapsed = time.time() - t0

        output = res.stdout
        loss_match = re.findall(r"loss[ =]+([\d\.]+)", output)
        last_loss = loss_match[-1] if loss_match else "N/A"

        ep1_match = re.search(r"epoch 1/2.*?([\d\.]+)s", output)
        ep2_match = re.search(r"epoch 2/2.*?([\d\.]+)s", output)
        ep1_time = float(ep1_match.group(1)) if ep1_match else 0.0
        ep2_time = float(ep2_match.group(1)) if ep2_match else 0.0

        val_match = re.findall(r"validation time:\s+([\d\.]+)s", output)
        val_time = float(val_match[-1]) if val_match else 0.0

        bench_results.append({
            "Config": name,
            "Total Time (s)": round(elapsed, 2),
            "Epoch 1 Time (s)": round(ep1_time, 2),
            "Epoch 2 Time (s)": round(ep2_time, 2),
            "Val Time (s)": round(val_time, 2),
            "Final Loss": last_loss,
            "Exit Code": res.returncode,
        })
        print(f"✓ {name:<30}: Ep1={ep1_time:5.2f}s | Ep2={ep2_time:5.2f}s | Val={val_time:5.2f}s | Total={elapsed:6.2f}s | Loss: {last_loss}")

    print("\n" + "=" * 85)
    print("RUNTIME BENCHMARK SUMMARY (MEASURED VALUES ONLY - 2 EPOCHS)")
    print("=" * 85)
    base_ep2 = bench_results[0]["Epoch 2 Time (s)"]
    for r in bench_results:
        steady_speedup = f"{base_ep2 / r['Epoch 2 Time (s)']:.2f}x" if (r['Epoch 2 Time (s)'] > 0 and base_ep2 > 0) else "N/A"
        print(f"{r['Config']:<30} | Ep1 (Warmup): {r['Epoch 1 Time (s)']:5.2f}s | Ep2 (Steady): {r['Epoch 2 Time (s)']:5.2f}s ({steady_speedup:<5}) | Val: {r['Val Time (s)']:5.2f}s | Total: {r['Total Time (s)']:6.2f}s | Loss: {r['Final Loss']}")
    print("=" * 85)
    print("Note: Epoch 1 includes cuDNN kernel autotuning and AMP initialization warm-up.")
    print("      Epoch 2 reflects steady-state training performance.")
    print("=" * 85)

# =====================================================================
# B. FOLD 0 ABLATION MODE ('fold0_ablation')
# =====================================================================
elif CONFIG_MODE == "fold0_ablation":
    if not RUN_ABLATION:
        print("=" * 85)
        print("[SAFETY GUARD] CONFIG_MODE is 'fold0_ablation' but RUN_ABLATION is False.")
        print("Estimated Cost: 3x Fold 0 runs (~45-60 min GPU time on Kaggle T4).")
        print("To confirm and run this ablation experiment, set RUN_ABLATION = True in Cell 4 and re-run.")
        print("=" * 85)
        raise SystemExit("Ablation guard active: set RUN_ABLATION = True to proceed.")

    print("=" * 85)
    print("RUNNING FOLD-0 SCIENTIFIC ABLATION EXPERIMENT")
    print("Estimated Cost: 3x Fold 0 training runs (~45-60 min GPU time on Kaggle T4)")
    print("WARNING: Fold 0 contains approximately 12 validation cases; Dice differences under ~0.03 are likely sampling noise.")
    print("=" * 85)

    base_ablation_dir = Path("/kaggle/working/ablation_results") if PLATFORM == "kaggle" else (OUT_DIR / "ablation_results")
    base_ablation_dir.mkdir(parents=True, exist_ok=True)

    shared_flags = [
        "--run-folds", "0",
        "--epochs", str(FULL_EPOCHS),
        "--folds", str(FULL_FOLDS),
        "--val-batch-size", str(VAL_BATCH_SIZE),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    if USE_AMP and torch.cuda.is_available():
        shared_flags.append("--amp")
    if USE_CUDNN_BENCHMARK and torch.cuda.is_available():
        shared_flags.append("--cudnn-benchmark")
    if USE_CHANNELS_LAST and torch.cuda.is_available():
        shared_flags.append("--channels-last-3d")

    ablation_configs = [
        (
            "(A) Baseline (Patient-Grouped Folds)",
            base_ablation_dir / "config_A_baseline",
            shared_flags + ["--out-dir", str(base_ablation_dir / "config_A_baseline")],
        ),
        (
            "(B) A + Resample & Augmentation",
            base_ablation_dir / "config_B_resample_aug",
            shared_flags + [
                "--resample-patches-each-epoch",
                "--augment",
                "--out-dir", str(base_ablation_dir / "config_B_resample_aug"),
            ],
        ),
        (
            "(C) B + Sampling Mix (lesion=0.5, bone=0.35, random=0.15)",
            base_ablation_dir / "config_C_sampling_mix",
            shared_flags + [
                "--resample-patches-each-epoch",
                "--augment",
                "--sampling-mix", "lesion=0.5,bone=0.35,random=0.15",
                "--out-dir", str(base_ablation_dir / "config_C_sampling_mix"),
            ],
        ),
    ]

    ablation_results = []
    for label, out_path, cmd_flags in ablation_configs:
        out_path.mkdir(parents=True, exist_ok=True)
        print(f"\n>>> Running Ablation {label} ...", flush=True)
        full_cmd = [sys.executable, str(script_to_run), str(DATASET_DIR)] + cmd_flags
        t0 = time.time()
        p = subprocess.run(full_cmd, capture_output=False, text=True)
        w_time = time.time() - t0
        if p.returncode != 0:
            print(f"[ERROR] Ablation run {label} failed with exit code {p.returncode}")
            continue

        metric_file = out_path / "metrics_fold_0.json"
        if not metric_file.exists():
            metric_file = out_path / "metrics.json"

        if metric_file.exists():
            try:
                m_data = json.loads(metric_file.read_text(encoding="utf-8"))
                ablation_results.append({
                    "Config": label,
                    "Bone Dice": m_data.get("bone_dice"),
                    "Edema Dice": m_data.get("bme_dice"),
                    "Lesion Sens": m_data.get("lesion_sensitivity"),
                    "FP/Case": m_data.get("fp_per_case"),
                    "Final Loss": m_data.get("final_loss"),
                    "AMP Skipped": m_data.get("amp_skipped_steps", 0),
                    "Wall Time": f"{w_time:.1f}s",
                })
            except Exception as e:
                print(f"Error reading metrics for {label}: {e}")

    if ablation_results:
        print("\n" + "=" * 105)
        print("FOLD 0 ABLATION RESULTS (SIDE-BY-SIDE MEASURED RUNTIME VALUES)")
        print("=" * 105)
        headers = ["Config", "Bone Dice", "Edema Dice", "Lesion Sens", "FP/Case", "Final Loss", "AMP Skipped", "Wall Time"]
        row_format = "{:<35} | {:<10} | {:<10} | {:<11} | {:<8} | {:<10} | {:<11} | {:<10}"
        print(row_format.format(*headers))
        print("-" * 105)
        for r in ablation_results:
            b_d = f"{r['Bone Dice']:.4f}" if isinstance(r['Bone Dice'], (int, float)) else str(r['Bone Dice'])
            e_d = f"{r['Edema Dice']:.4f}" if isinstance(r['Edema Dice'], (int, float)) else str(r['Edema Dice'])
            ls = f"{r['Lesion Sens']:.4f}" if isinstance(r['Lesion Sens'], (int, float)) else str(r['Lesion Sens'])
            fp = f"{r['FP/Case']:.2f}" if isinstance(r['FP/Case'], (int, float)) else str(r['FP/Case'])
            fl = f"{r['Final Loss']:.4f}" if isinstance(r['Final Loss'], (int, float)) else str(r['Final Loss'])
            print(row_format.format(
                r["Config"][:35],
                b_d,
                e_d,
                ls,
                fp,
                fl,
                str(r["AMP Skipped"]),
                str(r["Wall Time"]),
            ))
        print("=" * 105)
        print("[CAUTION] Fold 0 evaluates ~12 cases. Small Dice variations (e.g. < 0.03) are within expected stochastic fold noise.")
        print("=" * 105)

# =====================================================================
# C. TRAINING MODES ('quick_test', 'fold0', 'full')
# =====================================================================
else:
    # Resolve folds and epochs to run
    if CONFIG_MODE == "fold0":
        folds_to_run = [0]
        epochs_to_run = FULL_EPOCHS
    elif CONFIG_MODE == "quick_test":
        folds_to_run = [0, 1]
        epochs_to_run = 2
    else:  # full
        folds_to_run = RUN_FOLDS if RUN_FOLDS is not None else list(range(FULL_FOLDS))
        epochs_to_run = FULL_EPOCHS

    if SPECIFIC_FOLD is not None:
        folds_to_run = [SPECIFIC_FOLD]

    base_cmd_flags = [
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(OUT_DIR),
        "--epochs", str(epochs_to_run),
        "--folds", str(FULL_FOLDS),
        "--val-batch-size", str(VAL_BATCH_SIZE),
    ]
    if USE_AMP and torch.cuda.is_available():
        base_cmd_flags.append("--amp")
    if USE_CUDNN_BENCHMARK and torch.cuda.is_available():
        base_cmd_flags.append("--cudnn-benchmark")
    if USE_CHANNELS_LAST and torch.cuda.is_available():
        base_cmd_flags.append("--channels-last-3d")
    if PROFILE:
        base_cmd_flags.append("--profile")
    if RESUME:
        base_cmd_flags.append("--resume")
    if SKIP_COMPLETED:
        base_cmd_flags.append("--skip-completed")
    if CHECKPOINT_EVERY_EPOCH:
        base_cmd_flags.append("--checkpoint-every-epoch")
    if MAX_RUNTIME_MINUTES is not None:
        base_cmd_flags.extend(["--max-runtime-minutes", str(MAX_RUNTIME_MINUTES)])
    if globals().get("ENABLE_DATA_AUGMENTATION", False):
        base_cmd_flags.append("--augment")
    if globals().get("RESAMPLE_EACH_EPOCH", False):
        base_cmd_flags.append("--resample-patches-each-epoch")
    if globals().get("SAMPLING_MIX", None):
        base_cmd_flags.extend(["--sampling-mix", SAMPLING_MIX])
    if globals().get("MODEL_ARCH", None):
        base_cmd_flags.extend(["--model-arch", MODEL_ARCH])

    t_train_start = time.time()

    # Parallel multi-GPU scheduler when 2 GPUs visible and not FORCE_SEQUENTIAL
    if n_gpus >= 2 and not FORCE_SEQUENTIAL:
        print(f"[INFO] Scheduling {len(folds_to_run)} folds dynamically across 2 GPUs (T4 x2)...")
        log_dir = OUT_DIR / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)

        pending_folds = list(folds_to_run)
        gpu_slots = {0: None, 1: None}

        def launch_fold(fold_idx, gpu_id):
            log_path = log_dir / f"fold_{fold_idx}.log"
            log_f = open(log_path, "w", encoding="utf-8")
            cmd = [
                sys.executable, "-u", str(script_to_run), str(BASE_DIR),
                "--fold", str(fold_idx),
            ] + base_cmd_flags

            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

            t0 = time.time()
            print(f"-> Launching Fold {fold_idx} on GPU {gpu_id} (log: logs/fold_{fold_idx}.log)...", flush=True)
            p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
            return (fold_idx, p, log_f, t0)

        completed_folds = []
        while pending_folds or any(slot is not None for slot in gpu_slots.values()):
            for gpu_id in [0, 1]:
                if gpu_slots[gpu_id] is None and pending_folds:
                    next_f = pending_folds.pop(0)
                    gpu_slots[gpu_id] = launch_fold(next_f, gpu_id)

            time.sleep(2)
            for gpu_id in [0, 1]:
                slot = gpu_slots[gpu_id]
                if slot is not None:
                    f_idx, p, log_f, t0 = slot
                    ret = p.poll()
                    if ret is not None:
                        log_f.close()
                        elapsed = time.time() - t0
                        if ret != 0:
                            print(f"[ERROR] Fold {f_idx} on GPU {gpu_id} failed with exit code {ret}! Check logs/fold_{f_idx}.log")
                            raise subprocess.CalledProcessError(ret, f"Fold {f_idx}")
                        print(f"[OK] Fold {f_idx} finished successfully on GPU {gpu_id} in {int(elapsed // 60)}m {int(elapsed % 60)}s.", flush=True)
                        completed_folds.append(f_idx)
                        gpu_slots[gpu_id] = None

        print("\nAll scheduled folds finished successfully!")

    # Sequential execution on single GPU or CPU
    else:
        print(f"[INFO] Running sequentially on single device for folds: {folds_to_run}...")
        for fold_idx in folds_to_run:
            cmd = [
                sys.executable, "-u", str(script_to_run), str(BASE_DIR),
                "--fold", str(fold_idx),
            ] + base_cmd_flags
            print(f"--- Running Fold {fold_idx} ---")
            subprocess.run(cmd, check=True)

    # Merge per-fold metrics files into unified metrics.json
    print("\n--- Merging per-fold metrics into metrics.json ---")
    merge_cmd = [
        sys.executable, "-u", str(script_to_run), str(BASE_DIR),
        "--out-dir", str(OUT_DIR),
        "--folds", str(FULL_FOLDS),
        "--merge-metrics",
    ]
    subprocess.run(merge_cmd, check=False)

    # Check for incomplete folds and display resume banner if applicable
    incomplete_status_files = sorted(list(OUT_DIR.glob("status_fold_*.json")))
    incomplete_folds_info = []
    for sf in incomplete_status_files:
        try:
            st = json.loads(sf.read_text(encoding="utf-8"))
            if st.get("status") == "incomplete":
                incomplete_folds_info.append((st.get("fold"), st.get("epoch_reached"), st.get("target_epochs")))
        except Exception:
            pass

    has_partial = (OUT_DIR / "metrics_partial.json").exists()
    has_complete = (OUT_DIR / "metrics.json").exists()

    if incomplete_folds_info or (has_partial and not has_complete):
        print("\n" + "!" * 75)
        print("⚠️  TRAINING RUN INCOMPLETE (TIME GUARD OR PARTIAL EXECUTION)")
        print("!" * 75)
        for f_idx, ep, target_ep in incomplete_folds_info:
            print(f"  * Fold {f_idx}: Stopped at epoch {ep}/{target_ep}")
        print("\n  SAFEGUARD SUMMARY:")
        print("  - Final fold checkpoints (fold_k.pt) and metrics.json were NOT finalized.")
        print("  - Resumable checkpoint(s) (fold_k_latest.pt) and metrics_partial.json are safely stored.")
        print("\n  RESUME INSTRUCTIONS FOR NEXT KAGGLE RUN:")
        print("  1. In Kaggle, create a new notebook version or attach this run's output directory as input.")
        print("  2. In Cell 2 configuration, keep RESUME = True.")
        print("  3. Run the notebook. Training will resume seamlessly from the exact epoch reached.")
        print("!" * 75 + "\n")
    elif has_complete:
        print("[OK] All fold metrics merged into metrics.json!")


## 6b. [Alternative] Parallel Multi-GPU Training (Kaggle T4 x2)
On Kaggle sessions configured with **GPU T4 x2**, two 16GB GPUs are available.
This cell schedules all 5 folds dynamically across the 2 GPUs using a **per-GPU work queue**:
- When GPU 0 or GPU 1 becomes free, it immediately pulls the next pending fold from the queue.
- Both GPUs stay continuously utilized with zero idle time.
- Each fold logs independently to `OUT_DIR/logs/fold_<k>.log`.
- Checkpoints are saved independently to `OUT_DIR/fold_<k>.pt` and `OUT_DIR/metrics_fold_<k>.json`.
- At completion, it calls `--merge-metrics` to compile all fold results into `metrics.json` matching the sequential schema.
- If only 1 GPU is detected, it falls back to sequential execution automatically.


In [ ]:
RUN_ALTERNATIVE_CELL = False  # Set to True only if executing Cell 6b standalone
if not RUN_ALTERNATIVE_CELL:
    print("[INFO] Multi-GPU training already completed in Cell 6. Skipping duplicate Cell 6b.")
else:
    import os
    import subprocess
    import sys
    import time
    from pathlib import Path
    import torch
    
    script_to_run = working_script if working_script.exists() else source_script
    n_gpus = torch.cuda.device_count()
    print(f"[HARDWARE] Detected {n_gpus} CUDA GPU(s).")
    
    folds_to_run = [SPECIFIC_FOLD] if SPECIFIC_FOLD is not None else list(range(FULL_FOLDS))
    
    if n_gpus < 2:
        print("[INFO] Running sequentially on single GPU or CPU.")
        for fold in folds_to_run:
            cmd = [
                sys.executable, "-u", str(script_to_run), str(BASE_DIR),
                "--device", "cuda" if torch.cuda.is_available() else "cpu",
                "--out-dir", str(OUT_DIR),
                "--epochs", str(FULL_EPOCHS if CONFIG_MODE != "quick_test" else 2),
                "--folds", str(FULL_FOLDS),
                "--fold", str(fold),
                "--val-batch-size", str(VAL_BATCH_SIZE),
            ]
            if USE_AMP and torch.cuda.is_available():
                cmd.append("--amp")
            if USE_CUDNN_BENCHMARK and torch.cuda.is_available():
                cmd.append("--cudnn-benchmark")
            if USE_CHANNELS_LAST and torch.cuda.is_available():
                cmd.append("--channels-last-3d")
            if PROFILE:
                cmd.append("--profile")
            if RESUME:
                cmd.append("--resume")
            if globals().get("ENABLE_DATA_AUGMENTATION", False):
                cmd.append("--augment")
            if globals().get("RESAMPLE_EACH_EPOCH", False):
                cmd.append("--resample-patches-each-epoch")
            if globals().get("SAMPLING_MIX", None):
                cmd.extend(["--sampling-mix", SAMPLING_MIX])
            print(f"--- Running Fold {fold} ---")
            subprocess.run(cmd, check=True)
    else:
        print(f"[INFO] Scheduling {len(folds_to_run)} folds dynamically across 2 GPUs (T4 x2)...")
        log_dir = OUT_DIR / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        
        pending_folds = list(folds_to_run)
        gpu_slots = {0: None, 1: None}
        
        def launch_fold(fold, gpu_id):
            log_path = log_dir / f"fold_{fold}.log"
            log_f = open(log_path, "w", encoding="utf-8")
            cmd = [
                sys.executable, "-u", str(script_to_run), str(BASE_DIR),
                "--device", "cuda",
                "--out-dir", str(OUT_DIR),
                "--epochs", str(FULL_EPOCHS if CONFIG_MODE != "quick_test" else 2),
                "--folds", str(FULL_FOLDS),
                "--fold", str(fold),
                "--val-batch-size", str(VAL_BATCH_SIZE),
            ]
            if USE_AMP:
                cmd.append("--amp")
            if USE_CUDNN_BENCHMARK:
                cmd.append("--cudnn-benchmark")
            if USE_CHANNELS_LAST:
                cmd.append("--channels-last-3d")
            if PROFILE:
                cmd.append("--profile")
            if RESUME:
                cmd.append("--resume")
            if globals().get("ENABLE_DATA_AUGMENTATION", False):
                cmd.append("--augment")
            if globals().get("RESAMPLE_EACH_EPOCH", False):
                cmd.append("--resample-patches-each-epoch")
            if globals().get("SAMPLING_MIX", None):
                cmd.extend(["--sampling-mix", SAMPLING_MIX])
            
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
            
            t0 = time.time()
            print(f"-> Launching Fold {fold} on GPU {gpu_id} (log: logs/fold_{fold}.log)...", flush=True)
            p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
            return (fold, p, log_f, t0)
    
        # Dynamic dispatch loop
        completed_folds = []
        while pending_folds or any(slot is not None for slot in gpu_slots.values()):
            for gpu_id in [0, 1]:
                if gpu_slots[gpu_id] is None and pending_folds:
                    next_f = pending_folds.pop(0)
                    gpu_slots[gpu_id] = launch_fold(next_f, gpu_id)
            
            time.sleep(2)
            for gpu_id in [0, 1]:
                slot = gpu_slots[gpu_id]
                if slot is not None:
                    fold, p, log_f, t0 = slot
                    ret = p.poll()
                    if ret is not None:
                        log_f.close()
                        elapsed = time.time() - t0
                        if ret != 0:
                            print(f"[ERROR] Fold {fold} on GPU {gpu_id} failed with exit code {ret}! Check logs/fold_{fold}.log")
                            raise subprocess.CalledProcessError(ret, f"Fold {fold}")
                        print(f"[OK] Fold {fold} finished successfully on GPU {gpu_id} in {int(elapsed // 60)}m {int(elapsed % 60)}s.", flush=True)
                        completed_folds.append(fold)
                        gpu_slots[gpu_id] = None
    
        print("\nAll folds finished successfully!")
    
    # Merge per-fold metrics files into unified metrics.json
    print("\n--- Merging per-fold metrics into metrics.json ---")
    merge_cmd = [
        sys.executable, "-u", str(script_to_run), str(BASE_DIR),
        "--out-dir", str(OUT_DIR),
        "--folds", str(FULL_FOLDS),
        "--merge-metrics",
    ]
    subprocess.run(merge_cmd, check=True)
    print("[OK] All fold metrics merged into metrics.json!")
    

## 6c. Fold-0 Scientific Equivalence Check
Directly compares **Fold 0** run with:
1. **Baseline flags**: standard FP32, unbatched validation (`val_batch_size=1`), no cuDNN tuning.
2. **Optimized flags**: `--amp`, `--cudnn-benchmark`, `--channels-last-3d`, `--val-batch-size 8`.

Runs both under the **exact same seed (1337)** and prints:
- Bone Dice
- Edema (BME) Dice
- Lesion Sensitivity
- False-Positive Voxels per Case
- Final Training Loss
- Wall-Clock Time & Speedup Ratio


In [ ]:
import subprocess
import sys
import time
import json
import re
from pathlib import Path
import torch

# =====================================================================
# FOLD-0 SCIENTIFIC EQUIVALENCE CHECK (OFF BY DEFAULT)
# =====================================================================
RUN_EQUIVALENCE = False  # Set to True to run the Fold-0 equivalence check (Baseline vs Optimized)

if not RUN_EQUIVALENCE:
    print("=" * 75)
    print("FOLD-0 SCIENTIFIC EQUIVALENCE CHECK: OFF (Default)")
    print("=" * 75)
    print("Notice: This equivalence test runs Fold 0 TWICE (1 baseline run + 1 optimized run).")
    print("Estimated cost: ~2x the runtime of a single fold.")
    print("To run this equivalence check, explicitly set:")
    print("    RUN_EQUIVALENCE = True")
    print("=" * 75)
else:
    check_dir = Path("/kaggle/working/equiv_check") if PLATFORM == "kaggle" else (OUT_DIR / "equiv_check")
    check_dir.mkdir(parents=True, exist_ok=True)
    base_out = check_dir / "baseline"
    opt_out = check_dir / "optimized"
    base_out.mkdir(parents=True, exist_ok=True)
    opt_out.mkdir(parents=True, exist_ok=True)

    test_epochs = 2  # Micro-run for equivalence and timing comparison

    print("=" * 70)
    print(f"RUNNING FOLD-0 SCIENTIFIC EQUIVALENCE CHECK (Seed 1337, {test_epochs} Epochs)")
    print("=" * 70)

    # 1. Baseline Run
    print("\n[1/2] Running Fold 0 Baseline (FP32, sequential sliding window)...")
    cmd_base = [
        sys.executable, "-u", str(working_script), str(BASE_DIR),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(base_out),
        "--epochs", str(test_epochs),
        "--folds", "5",
        "--fold", "0",
        "--profile",
    ]
    t0 = time.time()
    res_base = subprocess.run(cmd_base, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time_base = time.time() - t0
    if res_base.returncode != 0:
        print(res_base.stdout)
        raise RuntimeError(f"Baseline run failed with code {res_base.returncode}")

    # 2. Optimized Run
    print("[2/2] Running Fold 0 Optimized (+AMP, +cuDNN, +channels_last, val_bs=8)...")
    cmd_opt = [
        sys.executable, "-u", str(working_script), str(BASE_DIR),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(opt_out),
        "--epochs", str(test_epochs),
        "--folds", "5",
        "--fold", "0",
        "--amp",
        "--cudnn-benchmark",
        "--channels-last-3d",
        "--val-batch-size", "8",
        "--profile",
    ]
    t0 = time.time()
    res_opt = subprocess.run(cmd_opt, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time_opt = time.time() - t0
    if res_opt.returncode != 0:
        print(res_opt.stdout)
        raise RuntimeError(f"Optimized run failed with code {res_opt.returncode}")

    # Parse metrics from metrics_fold_0.json
    m_base = json.loads((base_out / "metrics_fold_0.json").read_text(encoding="utf-8"))
    m_opt = json.loads((opt_out / "metrics_fold_0.json").read_text(encoding="utf-8"))

    # Parse final losses
    loss_base_m = re.findall(r"loss ([\d\.]+)", res_base.stdout)
    last_loss_base = float(loss_base_m[-1]) if loss_base_m else 0.0

    loss_opt_m = re.findall(r"loss ([\d\.]+)", res_opt.stdout)
    last_loss_opt = float(loss_opt_m[-1]) if loss_opt_m else 0.0

    print("\n" + "=" * 75)
    print("FOLD-0 SCIENTIFIC EQUIVALENCE & SPEEDUP RESULTS")
    print("=" * 75)
    print(f"{'Metric':<25} | {'Baseline (FP32)':<16} | {'Optimized (+AMP)':<16} | {'Diff / Ratio'}")
    print("-" * 75)
    print(f"{'Bone Dice':<25} | {m_base['bone_dice']:<16.4f} | {m_opt['bone_dice']:<16.4f} | {abs(m_base['bone_dice'] - m_opt['bone_dice']):+.4f}")
    print(f"{'Edema (BME) Dice':<25} | {m_base['bme_dice']:<16.4f} | {m_opt['bme_dice']:<16.4f} | {abs(m_base['bme_dice'] - m_opt['bme_dice']):+.4f}")
    print(f"{'Lesion Sensitivity':<25} | {m_base['lesion_sensitivity']:<16.4f} | {m_opt['lesion_sensitivity']:<16.4f} | {abs(m_base['lesion_sensitivity'] - m_opt['lesion_sensitivity']):+.4f}")
    print(f"{'False Positives / Case':<25} | {m_base['fp_per_case']:<16.2f} | {m_opt['fp_per_case']:<16.2f} | {abs(m_base['fp_per_case'] - m_opt['fp_per_case']):+.2f}")
    print(f"{'Final Training Loss':<25} | {last_loss_base:<16.4f} | {last_loss_opt:<16.4f} | {abs(last_loss_base - last_loss_opt):+.4f}")
    speedup = time_base / max(time_opt, 0.01)
    print(f"{'Wall-Clock Time':<25} | {time_base:<16.1f}s | {time_opt:<16.1f}s | {speedup:.2f}x faster")
    print("=" * 75)


## 6d. Speedup Benchmark: Measure Speed vs Scientific Baseline
Run a fast 1-fold × 2-epoch benchmark on your GPU comparing 5 configurations:
1. **Baseline**: standard PyTorch (FP32, no flags, val_batch_size=1)
2. **+AMP**: Automatic Mixed Precision (FP16)
3. **+AMP + cuDNN Benchmark**: + cuDNN heuristic autotuner
4. **+AMP + cuDNN + Channels Last**: + `channels_last_3d` memory format
5. **+All Speedups + Val Batching**: + `--val-batch-size 8`

Prints wall-clock time, profiling breakdown (`t_data`, `t_train`, `t_val`), and final loss for each configuration so you can verify numerical equivalence and measure real GPU gains. Checkpoints are kept in a separate temporary directory.


In [ ]:
RUN_SPEEDUP_BENCHMARK = False  # Set to True only if you explicitly want to run the 5-way GPU benchmark
if not RUN_SPEEDUP_BENCHMARK:
    print("===========================================================================")
    print("SPEEDUP BENCHMARK: OFF (Default) - Skipping to prevent extra GPU runtime")
    print("===========================================================================")
else:
    import subprocess
    import sys
    import time
    import json
    import re
    from pathlib import Path
    import torch
    
    bench_dir = Path("/kaggle/working/benchmark_results") if PLATFORM == "kaggle" else (OUT_DIR / "benchmark_results")
    bench_dir.mkdir(parents=True, exist_ok=True)
    
    configs = [
        ("Baseline (FP32, val_bs=1)", []),
        ("+AMP (FP16)", ["--amp"]),
        ("+AMP + cuDNN Benchmark", ["--amp", "--cudnn-benchmark"]),
        ("+AMP + cuDNN + Channels-Last", ["--amp", "--cudnn-benchmark", "--channels-last-3d"]),
        ("+All Speedups (inc. Val BS=8)", ["--amp", "--cudnn-benchmark", "--channels-last-3d", "--val-batch-size", "8"]),
    ]
    
    results = []
    
    print("=" * 75)
    print(f"RUNNING SPEEDUP BENCHMARK (1 Fold x 2 Epochs, Device: {'CUDA' if torch.cuda.is_available() else 'CPU'})")
    print("=" * 75)
    
    for name, flags in configs:
        sub_dir = bench_dir / name.replace(" ", "_").replace("+", "p_").replace("(", "").replace(")", "").replace("=", "_").replace(",", "_")
        sub_dir.mkdir(parents=True, exist_ok=True)
        
        cmd = [
            sys.executable, "-u", str(working_script), str(BASE_DIR),
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
            "--out-dir", str(sub_dir),
            "--epochs", "2",
            "--folds", "5",
            "--fold", "0",
            "--profile",
        ] + flags
        
        t0 = time.time()
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        elapsed = time.time() - t0
        
        output = res.stdout
        loss_match = re.findall(r"loss ([\d\.]+)", output)
        last_loss = loss_match[-1] if loss_match else "N/A"
        
        val_time_match = re.findall(r"validation time:\s+([\d\.]+)s", output)
        val_time = float(val_time_match[-1]) if val_time_match else 0.0
        
        results.append({
            "Config": name,
            "Total Time (s)": round(elapsed, 1),
            "Val Time (s)": round(val_time, 1),
            "Final Loss": last_loss,
            "Exit Code": res.returncode,
        })
        print(f"✓ {name:<35}: {elapsed:6.1f}s | Val: {val_time:5.1f}s | Loss: {last_loss}")
    
    print("\n" + "=" * 75)
    print("BENCHMARK SUMMARY")
    print("=" * 75)
    base_time = results[0]["Total Time (s)"]
    for r in results:
        speedup = f"{base_time / r['Total Time (s)']:.2f}x" if r['Total Time (s)'] > 0 else "N/A"
        print(f"{r['Config']:<35} | Total: {r['Total Time (s)']:5.1f}s ({speedup:<5}) | Val: {r['Val Time (s)']:5.1f}s | Loss: {r['Final Loss']}")
    print("=" * 75)
    

## 6e. [OPT-IN DATA EXPERIMENT] Resampling & 3D Augmentation Comparison
> **OPT-IN EXPERIMENT (OFF by default)**: Keeps baseline 100% untouched when off.
>
> **Motivation**: In baseline training, 384 fixed patches per fold (8 per case) are sampled once at startup
> and reused across all 40 epochs with zero data augmentation. This creates severe data starvation for
> rare bone marrow edema (BME) lesions.
>
> When enabled, this cell runs Fold 0 twice:
> 1. **Baseline**: static 384 patches, no augmentation.
> 2. **Data Experiment**: `--resample-patches-each-epoch` + `--augment` (flip, intensity scale/shift, Gaussian noise).
> Outputs are strictly isolated outside `data/` into `/kaggle/working/data_exp_baseline` and `/kaggle/working/data_exp_augmented`.
> Prints side-by-side measured runtime metrics (Dice, sensitivity, FP/case, loss, time, AMP non-finite steps).

In [ ]:
# =====================================================================
# OPT-IN DATA EXPERIMENT: RESAMPLING & 3D AUGMENTATION (OFF BY DEFAULT)
# =====================================================================
RUN_EXPERIMENT_DATA = False  # Set to True to run the Fold-0 data experiment comparison

if RUN_EXPERIMENT_DATA:
    import subprocess
    import sys
    import time
    import json
    import re
    from pathlib import Path
    import torch

    exp_root = Path("/kaggle/working/data_exp") if PLATFORM == "kaggle" else (OUT_DIR.parent / "data_exp")
    exp_root.mkdir(parents=True, exist_ok=True)
    exp_base_dir = exp_root / "baseline_f0"
    exp_aug_dir = exp_root / "augmented_f0"
    exp_base_dir.mkdir(parents=True, exist_ok=True)
    exp_aug_dir.mkdir(parents=True, exist_ok=True)

    exp_epochs = 2 if CONFIG_MODE == "quick_test" else FULL_EPOCHS

    print("=" * 75)
    print(f"RUNNING FOLD-0 DATA EXPERIMENT COMPARISON ({exp_epochs} Epochs, Seed 1337)")
    print("=" * 75)

    # 1. Baseline Fold 0 (Static patches, no augmentation)
    print("\n[1/2] Running Fold 0 Baseline (Static patches, no augmentation)...")
    cmd_base = [
        sys.executable, "-u", str(working_script), str(BASE_DIR),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(exp_base_dir),
        "--epochs", str(exp_epochs),
        "--folds", "5",
        "--fold", "0",
        "--amp",
        "--cudnn-benchmark",
        "--channels-last-3d",
        "--val-batch-size", "8",
        "--profile",
    ]
    t0 = time.time()
    res_base = subprocess.run(cmd_base, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time_base = time.time() - t0
    if res_base.returncode != 0:
        print(res_base.stdout)
        raise RuntimeError(f"Data experiment baseline failed with code {res_base.returncode}")

    # 2. Data Experiment Fold 0 (Per-epoch resampling + 3D augmentations)
    print("[2/2] Running Fold 0 Data Experiment (+Resampling each epoch, +Augmentation)...")
    cmd_aug = [
        sys.executable, "-u", str(working_script), str(BASE_DIR),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(exp_aug_dir),
        "--epochs", str(exp_epochs),
        "--folds", "5",
        "--fold", "0",
        "--amp",
        "--cudnn-benchmark",
        "--channels-last-3d",
        "--val-batch-size", "8",
        "--resample-patches-each-epoch",
        "--augment",
        "--profile",
    ]
    t0 = time.time()
    res_aug = subprocess.run(cmd_aug, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time_aug = time.time() - t0
    if res_aug.returncode != 0:
        print(res_aug.stdout)
        raise RuntimeError(f"Data experiment augmented run failed with code {res_aug.returncode}")

    # Parse metrics
    m_base = json.loads((exp_base_dir / "metrics_fold_0.json").read_text(encoding="utf-8"))
    m_aug = json.loads((exp_aug_dir / "metrics_fold_0.json").read_text(encoding="utf-8"))

    loss_base_m = re.findall(r"loss ([\d\.]+)", res_base.stdout)
    last_loss_base = float(loss_base_m[-1]) if loss_base_m else 0.0
    loss_aug_m = re.findall(r"loss ([\d\.]+)", res_aug.stdout)
    last_loss_aug = float(loss_aug_m[-1]) if loss_aug_m else 0.0

    print("\n" + "=" * 78)
    print("FOLD-0 DATA EXPERIMENT MEASURED RESULTS (STATIC vs RESAMPLE+AUGMENT)")
    print("=" * 78)
    print(f"{'Metric':<25} | {'Baseline (Static)':<18} | {'Resample + Aug':<18} | {'Delta'}")
    print("-" * 78)
    print(f"{'Bone Dice':<25} | {m_base['bone_dice']:<18.4f} | {m_aug['bone_dice']:<18.4f} | {m_aug['bone_dice'] - m_base['bone_dice']:+.4f}")
    print(f"{'Edema (BME) Dice':<25} | {m_base['bme_dice']:<18.4f} | {m_aug['bme_dice']:<18.4f} | {m_aug['bme_dice'] - m_base['bme_dice']:+.4f}")
    print(f"{'Lesion Sensitivity':<25} | {m_base['lesion_sensitivity']:<18.4f} | {m_aug['lesion_sensitivity']:<18.4f} | {m_aug['lesion_sensitivity'] - m_base['lesion_sensitivity']:+.4f}")
    print(f"{'False Positives / Case':<25} | {m_base['fp_per_case']:<18.2f} | {m_aug['fp_per_case']:<18.2f} | {m_aug['fp_per_case'] - m_base['fp_per_case']:+.2f}")
    print(f"{'Final Training Loss':<25} | {last_loss_base:<18.4f} | {last_loss_aug:<18.4f} | {last_loss_aug - last_loss_base:+.4f}")
    print(f"{'AMP Skipped Steps':<25} | {m_base.get('amp_skipped_steps', 0):<18} | {m_aug.get('amp_skipped_steps', 0):<18} | {m_aug.get('amp_skipped_steps', 0) - m_base.get('amp_skipped_steps', 0):+d}")
    print(f"{'Wall-Clock Time':<25} | {time_base:<18.1f}s | {time_aug:<18.1f}s | {time_aug - time_base:+.1f}s")
    print("=" * 78)
else:
    print("Data experiment mode is OFF (RUN_EXPERIMENT_DATA = False).")
    print("Baseline training uses static 384 patches per fold with zero augmentation.")


## [EXPERIMENTAL] Hyperparameter Changes (OFF by default - Changes Experiment)
> **CAUTION**: The options below modify the scientific experiment and training dynamics. They are **OFF by default** and must **NOT** be used for the baseline benchmark unless you explicitly want to compare different model dynamics.
> 
> 1. `--batch-size 4`: Samples 4 patches per gradient step instead of 1. Changes optimizer step count per epoch from 488 to 122 and alters gradient variance.
> 2. `--val-stride 96,96,12`: Uses a non-overlapping sliding-window stride during final validation instead of `48,48,6`. Samples ~45 windows instead of ~364 per case, speeding up validation at the cost of less patch averaging.


In [ ]:
# =====================================================================
# EXPERIMENTAL TRAINING DEMONSTRATION (OFF BY DEFAULT)
# =====================================================================
RUN_EXPERIMENTAL = False  # Keep FALSE to avoid changing scientific baseline!

if RUN_EXPERIMENTAL:
    print("[WARNING] Running experimental training with altered batch size and stride!")
    exp_out = OUT_DIR / "experimental_bs4"
    exp_cmd = [
        sys.executable, "-u", str(working_script), str(BASE_DIR),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out-dir", str(exp_out),
        "--epochs", "40",
        "--folds", "5",
        "--amp",
        "--cudnn-benchmark",
        "--channels-last-3d",
        "--batch-size", "4",            # [EXPERIMENT CHANGE] Larger batch size
        "--val-stride", "96,96,12",     # [EXPERIMENT CHANGE] Coarser validation stride
    ]
    subprocess.run(exp_cmd, check=True)
else:
    print("Experimental mode is OFF. Baseline experiment hyperparameters preserved.")


## 7. Cross-Validation Results & Dice Summary
Reads and summarizes patient-level cross-validation scores saved in `metrics.json`.

In [ ]:
import json
from pathlib import Path

metrics_file = OUT_DIR / "metrics.json"
partial_file = OUT_DIR / "metrics_partial.json"

if metrics_file.exists():
    data = json.loads(metrics_file.read_text(encoding="utf-8"))
    print("=" * 60)
    print("3D U-NET CROSS-VALIDATION SUMMARY (ALL FOLDS)")
    print("=" * 60)
    summary = data.get("summary", {})
    for metric, stats in summary.items():
        if stats:
            print(f"{metric:<22}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print("-" * 60)
    print(f"Edema presence correct : {data.get('presence_correct')}/{data.get('presence_n')} cases")
    print("=" * 60)
    print("\nPer-Fold Scores:")
    for row in data.get("per_fold", []):
        print(f"  Fold {row['fold']}: Bone Dice={row['bone_dice']:.3f}, BME Dice={row['bme_dice']:.3f}, "
              f"Sens={row['lesion_sensitivity']:.3f}, FP/case={row['fp_per_case']:.2f}")
elif partial_file.exists():
    rows = json.loads(partial_file.read_text(encoding="utf-8"))
    print("Partial fold metrics recorded so far:")
    for row in rows:
        print(f"  Fold {row['fold']}: Bone Dice={row['bone_dice']:.3f}, BME Dice={row['bme_dice']:.3f}, "
              f"Sens={row['lesion_sensitivity']:.3f}, FP/case={row['fp_per_case']:.2f}")
else:
    print("No metrics recorded yet. Check training logs above.")


In [ ]:
# =====================================================================
# AUTOMATIC PACKAGING & MEASURED TIMING BREAKDOWN
# =====================================================================
import json
import time
import zipfile
from pathlib import Path

t_pack_start = time.time()
zip_dest = Path("/kaggle/working/results3d.zip") if PLATFORM == "kaggle" else (OUT_DIR / "results3d.zip")
pt_files = sorted(OUT_DIR.glob("*.pt"))

if pt_files:
    with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for pt in pt_files:
            zf.write(pt, arcname=pt.name)
        metrics_f = OUT_DIR / "metrics.json"
        if metrics_f.exists():
            zf.write(metrics_f, arcname="metrics.json")
    t_pack_time = time.time() - t_pack_start
    print("=" * 70)
    print(f"✓ Packaged {len(pt_files)} model weight(s) into: {zip_dest}")
    print(f"  Archive size: {zip_dest.stat().st_size / (1024*1024):.2f} MB")
    print("=" * 70)
else:
    t_pack_time = time.time() - t_pack_start
    print("[WARNING] No checkpoint files (*.pt) found in OUT_DIR to package.")

# Print End-to-End Measured Timing Breakdown
print("\n" + "=" * 70)
print("COMPREHENSIVE RUNTIME TIMING BREAKDOWN (MEASURED VALUES ONLY)")
print("=" * 70)
if 't_install_time' in globals():
    print(f"  1. Dependencies check / pip install : {t_install_time:6.2f}s")
if 't_discovery_time' in globals():
    print(f"  2. Dataset discovery (read-only)    : {t_discovery_time:6.2f}s")

metrics_file = OUT_DIR / "metrics.json"
if metrics_file.exists():
    try:
        m_data = json.loads(metrics_file.read_text(encoding="utf-8"))
        tm = m_data.get("timing", {})
        if tm:
            print(f"  3. Volume loading (disk -> RAM)     : {tm.get('volume_load_s', 0):6.2f}s")
            print(f"  4. Patch sampling (static extraction): {tm.get('patch_sampling_s', 0):6.2f}s")
            print(f"  5. Training forward/backward (all)  : {tm.get('train_s', 0):6.2f}s")
            print(f"  6. Validation inference (sliding win): {tm.get('val_s', 0):6.2f}s")
            print(f"  7. Checkpoint saving (torch.save)   : {tm.get('save_s', 0):6.2f}s")
            print(f"  -------------------------------------------------------------")
            print(f"  Subtotal: 3D U-Net pipeline         : {tm.get('total_s', 0):6.2f}s")
    except Exception:
        pass

print(f"  8. Results packaging (.zip export)  : {t_pack_time:6.2f}s")
print("=" * 70)


# =====================================================================
# RUN_STATUS.JSON GENERATION & OUTPUT DIRECTORY CLEANUP
# =====================================================================
import re

completed_folds = []
for pt in OUT_DIR.glob("fold_*.pt"):
    m = re.search(r"fold_(\d+)\.pt", pt.name)
    if m:
        completed_folds.append(int(m.group(1)))
completed_folds.sort()

expected_folds = [0, 1, 2, 3, 4] if CONFIG_MODE == "full" else ([0] if CONFIG_MODE == "fold0" else [0, 1])
has_metrics = (OUT_DIR / "metrics.json").exists()
is_complete = set(expected_folds).issubset(set(completed_folds)) and has_metrics

timings_dict = {}
if "t_install_time" in globals():
    timings_dict["install_s"] = round(t_install_time, 2)
if "t_discovery_time" in globals():
    timings_dict["discovery_s"] = round(t_discovery_time, 2)
if "t_pack_time" in globals():
    timings_dict["package_s"] = round(t_pack_time, 2)
if (OUT_DIR / "metrics.json").exists():
    try:
        m_json = json.loads((OUT_DIR / "metrics.json").read_text(encoding="utf-8"))
        timings_dict.update(m_json.get("timing", {}))
    except Exception:
        pass

fp = str(total_cases) if "total_cases" in globals() else "bme_dataset"
run_status_payload = {
    "mode": CONFIG_MODE,
    "fingerprint": fp,
    "completed_folds": completed_folds,
    "complete": bool(is_complete),
    "incomplete_reason": "" if is_complete else "Expected fold checkpoints or metrics missing",
    "per_phase_timings": timings_dict,
    "written_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

status_dest = Path("/kaggle/working/run_status.json") if PLATFORM == "kaggle" else (OUT_DIR / "run_status.json")
status_dest.write_text(json.dumps(run_status_payload, indent=2), encoding="utf-8")
print(f"✓ Generated run_status.json: complete={is_complete}, folds={completed_folds}")

# Clean up any non-output temp directories in /kaggle/working to keep output small
if PLATFORM == "kaggle":
    kw = Path("/kaggle/working")
    for stray in [kw / "bme_dataset", kw / "data_exp", kw / "benchmark_results", kw / "ablation_results", kw / "equiv_check"]:
        if stray.exists():
            shutil.rmtree(stray, ignore_errors=True)
    
    total_kw_bytes = sum(f.stat().st_size for f in kw.rglob("*") if f.is_file())
    print(f"✓ Final /kaggle/working total output size: {total_kw_bytes / (1024*1024):.2f} MB")


In [ ]:
# =====================================================================
# OPT-IN EVALUATION SUITE & DIAGNOSTICS (eval_suite)
# =====================================================================
import base64
import hashlib
import os
import subprocess
import sys
from pathlib import Path

EVAL_SUITE_B64 = """..."""
EVAL_SUITE_EXPECTED_SHA = "9bc073cb650f599a931c995c7ecaabea3b98fa57a16ad4bcffbc1e4ebde5eb97"

POSTPROCESS_B64 = """IiIiU2hhcmVkIHBvc3QtcHJvY2Vzc2luZyBmb3IgM0QgQm9uZSBhbmQgRWRlbWEgc2VnbWVudGF0aW9uLgoKRmVhdHVyZXM6CjEuIFByb2JhYmlsaXR5IGF2ZXJhZ2luZyAoc29mdCBjb25zZW5zdXMpIHZzIGRpc2NyZXRlIHBhdGNoIHZvdGluZyAoYmFzZWxpbmUpLgoyLiBCb25lIGdhdGluZzogZWRlbWEgcHJlZGljdGVkIG91dHNpZGUgYm9uZSBpcyBzdHJpY3RseSBzdXBwcmVzc2VkLgozLiBNaW5pbXVtIGNvbXBvbmVudCBzaXplIGZpbHRlciBpbiBtbV4zIHVzaW5nIHJlYWwgc2NhbiB2b3hlbCBzcGFjaW5nLgo0LiBSZXR1cm5zIGNsZWFuIHVpbnQ4IGxhYmVsbWFwICgwPWJhY2tncm91bmQsIDE9Ym9uZSwgMj1lZGVtYSkgYW5kIHJlbW92YWwgZGlhZ25vc3RpY3MuCjUuIEluLXBsYW5lIGJvbmUgbWFzayBkaWxhdGlvbiBhbmQgb3JhY2xlIGJvbmUgb3ZlcnJpZGUgZm9yIGRpYWdub3N0aWMgZXZhbHVhdGlvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHNjaXB5Lm5kaW1hZ2UgYXMgbmRpCmZyb20gc2NpcHkubmRpbWFnZSBpbXBvcnQgbGFiZWwgYXMgY2NfbGFiZWwKCgpkZWYgaW5fcGxhbmVfZGlsYXRlKG1hc2s6IG5wLm5kYXJyYXksIHZveGVsczogaW50ID0gMSkgLT4gbnAubmRhcnJheToKICAgICIiIkRpbGF0ZSBtYXNrIGluLXBsYW5lIChheGVzIDAgYW5kIDEpIGxlYXZpbmcgdGhyb3VnaC1wbGFuZSAoYXhpcyAyKSB1bnRvdWNoZWQuIiIiCiAgICBpZiB2b3hlbHMgPD0gMDoKICAgICAgICByZXR1cm4gbWFzawogICAgc3RydWN0ID0gbnAuemVyb3MoKDMsIDMsIDMpLCBkdHlwZT1ib29sKQogICAgc3RydWN0WzosIDosIDFdID0gbmRpLmdlbmVyYXRlX2JpbmFyeV9zdHJ1Y3R1cmUoMiwgMSkKICAgIHJldHVybiBuZGkuYmluYXJ5X2RpbGF0aW9uKG1hc2ssIHN0cnVjdHVyZT1zdHJ1Y3QsIGl0ZXJhdGlvbnM9dm94ZWxzKQoKCmRlZiBwb3N0cHJvY2Vzc192b2x1bWUoCiAgICAqLAogICAgcHJvYl9ib25lOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCiAgICBwcm9iX2JtZTogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgdm90ZXNfYm9uZTogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgdm90ZXNfYm1lOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCiAgICBoaXRzOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCiAgICB0aHJlc2hvbGQ6IGZsb2F0ID0gMC41LAogICAgYm9uZV90aHJlc2hvbGQ6IGZsb2F0ID0gMC41LAogICAgbWluX3NpemVfbW0zOiBmbG9hdCA9IDAuMCwKICAgIHNwYWNpbmc6IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdID0gKDEuMCwgMS4wLCAxLjApLAogICAgcHJvYl9hdmVyYWdlOiBib29sID0gVHJ1ZSwKICAgIGJvbmVfbWFza19vdmVycmlkZTogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgZGlsYXRlX2JvbmVfaW5fcGxhbmU6IGludCA9IDAsCiAgICBza2lwX2JvbmVfZ2F0aW5nOiBib29sID0gRmFsc2UsCikgLT4gdHVwbGVbbnAubmRhcnJheSwgaW50XToKICAgICIiIlBvc3QtcHJvY2VzcyBwcmVkaWN0ZWQgYm9uZSBhbmQgZWRlbWEgdm9sdW1lcy4KCiAgICBBcmdzOgogICAgICAgIHByb2JfYm9uZTogQ29udGludW91cyBhdmVyYWdlIHByb2JhYmlsaXR5IHZvbHVtZSBmb3IgYm9uZSAoMC4wIHRvIDEuMCkuCiAgICAgICAgcHJvYl9ibWU6IENvbnRpbnVvdXMgYXZlcmFnZSBwcm9iYWJpbGl0eSB2b2x1bWUgZm9yIGVkZW1hICgwLjAgdG8gMS4wKS4KICAgICAgICB2b3Rlc19ib25lOiBBY2N1bXVsYXRlZCBwYXRjaCB2b3RlcyBmb3IgYm9uZS4KICAgICAgICB2b3Rlc19ibWU6IEFjY3VtdWxhdGVkIHBhdGNoIHZvdGVzIGZvciBlZGVtYS4KICAgICAgICBoaXRzOiBOdW1iZXIgb2YgcGF0Y2ggb3ZlcmxhcHBpbmcgaGl0cyBwZXIgdm94ZWwuCiAgICAgICAgdGhyZXNob2xkOiBEZWNpc2lvbiB0aHJlc2hvbGQgZm9yIGVkZW1hIChkZWZhdWx0IDAuNSkuCiAgICAgICAgYm9uZV90aHJlc2hvbGQ6IERlY2lzaW9uIHRocmVzaG9sZCBmb3IgYm9uZSAoZGVmYXVsdCAwLjUpLgogICAgICAgIG1pbl9zaXplX21tMzogTWluaW11bSBjb25uZWN0ZWQgY29tcG9uZW50IHZvbHVtZSBpbiBtbV4zIHRvIGtlZXAuCiAgICAgICAgc3BhY2luZzogVm94ZWwgc3BhY2luZyAoZHgsIGR5LCBkeikgaW4gbW0uCiAgICAgICAgcHJvYl9hdmVyYWdlOiBJZiBUcnVlLCB1c2VzIGNvbnRpbnVvdXMgcHJvYmFiaWxpdHkgdm9sdW1lczsKICAgICAgICAgICAgICAgICAgICAgIGlmIEZhbHNlLCB1c2VzIGJpbmFyaXplLXRoZW4tdm90ZSBiYXNlbGluZSBjb25zZW5zdXMuCiAgICAgICAgYm9uZV9tYXNrX292ZXJyaWRlOiBPcHRpb25hbCBleHBsaWNpdCBib29sZWFuIGJvbmUgbWFzayAoZm9yIG9yYWNsZSBkaWFnbm9zdGljcykuCiAgICAgICAgZGlsYXRlX2JvbmVfaW5fcGxhbmU6IE51bWJlciBvZiB2b3hlbHMgdG8gZGlsYXRlIHRoZSBib25lIG1hc2sgaW4tcGxhbmUuCiAgICAgICAgc2tpcF9ib25lX2dhdGluZzogSWYgVHJ1ZSwgZWRlbWEgaXMgbm90IG1hc2tlZCBieSBib25lLgoKICAgIFJldHVybnM6CiAgICAgICAgKG91dF9sYWJlbG1hcCwgbGVzaW9uc19yZW1vdmVkX2NvdW50KQogICAgICAgIG91dF9sYWJlbG1hcCBpcyB1aW50OCB3aXRoIDA9YmcsIDE9Ym9uZSwgMj1lZGVtYS4KICAgICIiIgogICAgaWYgcHJvYl9hdmVyYWdlOgogICAgICAgIGlmIHByb2JfYm9uZSBpcyBOb25lIG9yIHByb2JfYm1lIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByb2JfYm9uZSBhbmQgcHJvYl9ibWUgbXVzdCBiZSBwcm92aWRlZCB3aGVuIHByb2JfYXZlcmFnZT1UcnVlIikKICAgICAgICBzaGFwZSA9IHByb2JfYm9uZS5zaGFwZQogICAgICAgIGlmIGJvbmVfbWFza19vdmVycmlkZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgYm9uZV9tYXNrID0gYm9uZV9tYXNrX292ZXJyaWRlLmFzdHlwZShib29sKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJvbmVfbWFzayA9IHByb2JfYm9uZSA+PSBib25lX3RocmVzaG9sZAogICAgICAgIGlmIGRpbGF0ZV9ib25lX2luX3BsYW5lID4gMDoKICAgICAgICAgICAgYm9uZV9tYXNrID0gaW5fcGxhbmVfZGlsYXRlKGJvbmVfbWFzaywgZGlsYXRlX2JvbmVfaW5fcGxhbmUpCgogICAgICAgIGlmIHNraXBfYm9uZV9nYXRpbmc6CiAgICAgICAgICAgIGJtZV9tYXNrID0gcHJvYl9ibWUgPj0gdGhyZXNob2xkCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYm1lX21hc2sgPSAocHJvYl9ibWUgPj0gdGhyZXNob2xkKSAmIGJvbmVfbWFzawogICAgZWxzZToKICAgICAgICBpZiB2b3Rlc19ib25lIGlzIE5vbmUgb3Igdm90ZXNfYm1lIGlzIE5vbmUgb3IgaGl0cyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ2b3Rlc19ib25lLCB2b3Rlc19ibWUsIGFuZCBoaXRzIG11c3QgYmUgcHJvdmlkZWQgd2hlbiBwcm9iX2F2ZXJhZ2U9RmFsc2UiKQogICAgICAgIHNoYXBlID0gdm90ZXNfYm9uZS5zaGFwZQogICAgICAgIGggPSBucC5tYXhpbXVtKGhpdHMsIDEpCiAgICAgICAgaWYgYm9uZV9tYXNrX292ZXJyaWRlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBib25lX21hc2sgPSBib25lX21hc2tfb3ZlcnJpZGUuYXN0eXBlKGJvb2wpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYm9uZV9tYXNrID0gKHZvdGVzX2JvbmUgKiAyKSA+PSBoCiAgICAgICAgaWYgZGlsYXRlX2JvbmVfaW5fcGxhbmUgPiAwOgogICAgICAgICAgICBib25lX21hc2sgPSBpbl9wbGFuZV9kaWxhdGUoYm9uZV9tYXNrLCBkaWxhdGVfYm9uZV9pbl9wbGFuZSkKCiAgICAgICAgaWYgc2tpcF9ib25lX2dhdGluZzoKICAgICAgICAgICAgYm1lX21hc2sgPSAodm90ZXNfYm1lICogMikgPj0gaAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJtZV9tYXNrID0gKCh2b3Rlc19ibWUgKiAyKSA+PSBoKSAmIGJvbmVfbWFzawoKICAgICMgU2l6ZSBmaWx0ZXJpbmcgaW4gbW1eMwogICAgbl9yZW1vdmVkID0gMAogICAgaWYgbWluX3NpemVfbW0zID4gMC4wIGFuZCBibWVfbWFzay5hbnkoKToKICAgICAgICB2b3hlbF92b2wgPSBmbG9hdChhYnMoc3BhY2luZ1swXSAqIHNwYWNpbmdbMV0gKiBzcGFjaW5nWzJdKSkKICAgICAgICBsYWJlbGVkX2JtZSwgbnVtX2ZlYXR1cmVzID0gY2NfbGFiZWwoYm1lX21hc2spCiAgICAgICAgaWYgbnVtX2ZlYXR1cmVzID4gMDoKICAgICAgICAgICAgY291bnRzID0gbnAuYmluY291bnQobGFiZWxlZF9ibWUucmF2ZWwoKSkKICAgICAgICAgICAga2VlcF9tYXNrID0gbnAuemVyb3MobnVtX2ZlYXR1cmVzICsgMSwgZHR5cGU9Ym9vbCkKICAgICAgICAgICAgZm9yIGNvbXBfaWR4IGluIHJhbmdlKDEsIG51bV9mZWF0dXJlcyArIDEpOgogICAgICAgICAgICAgICAgdm9sX21tMyA9IGNvdW50c1tjb21wX2lkeF0gKiB2b3hlbF92b2wKICAgICAgICAgICAgICAgIGlmIHZvbF9tbTMgPj0gbWluX3NpemVfbW0zOgogICAgICAgICAgICAgICAgICAgIGtlZXBfbWFza1tjb21wX2lkeF0gPSBUcnVlCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIG5fcmVtb3ZlZCArPSAxCiAgICAgICAgICAgIGJtZV9tYXNrID0ga2VlcF9tYXNrW2xhYmVsZWRfYm1lXQoKICAgICMgQXNzZW1ibGUgb3V0cHV0IHZvbHVtZQogICAgb3V0ID0gbnAuemVyb3Moc2hhcGUsIGR0eXBlPW5wLnVpbnQ4KQogICAgb3V0W2JvbmVfbWFza10gPSAxCiAgICBvdXRbYm1lX21hc2tdID0gMgoKICAgIHJldHVybiBvdXQsIG5fcmVtb3ZlZAo="""
POSTPROCESS_EXPECTED_SHA = "2f232e73e060babe4141fdc4ee1c38a421f4577d73ff900f6f4b738a71488b3e"

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
eval_script = working_dir / "eval_suite.py"
postprocess_script = working_dir / "postprocess.py"

# Extract embedded scripts
eval_bytes = base64.b64decode(EVAL_SUITE_B64)
eval_sha = hashlib.sha256(eval_bytes).hexdigest()
if eval_sha != EVAL_SUITE_EXPECTED_SHA:
    print(f"[WARNING] eval_suite.py SHA-256 mismatch! Expected {EVAL_SUITE_EXPECTED_SHA[:10]}... got {eval_sha[:10]}...")
eval_script.write_bytes(eval_bytes)

postprocess_bytes = base64.b64decode(POSTPROCESS_B64)
postprocess_sha = hashlib.sha256(postprocess_bytes).hexdigest()
if postprocess_sha != POSTPROCESS_EXPECTED_SHA:
    print(f"[WARNING] postprocess.py SHA-256 mismatch! Expected {POSTPROCESS_EXPECTED_SHA[:10]}... got {postprocess_sha[:10]}...")
postprocess_script.write_bytes(postprocess_bytes)

print(f"[OK] Extracted eval_suite.py (SHA: {eval_sha[:8]}...) and postprocess.py (SHA: {postprocess_sha[:8]}...)")

if CONFIG_MODE == "eval_suite":
    print("=" * 60)
    print("CONFIG_MODE: eval_suite selected")
    if not RUN_EVAL:
        print("[SAFETY NOTICE] RUN_EVAL is False. Set RUN_EVAL = True to execute the evaluation suite.")
    else:
        print("Executing Evaluation Suite & Diagnostics...")
        out_dir = working_dir / "eval_results"
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable,
            str(eval_script),
            "--out-dir", str(out_dir),
            "--synthetic"
        ]
        print("Running command:", " ".join(cmd))
        subprocess.run(cmd, check=True)
